In [1]:
!pip install torch transformers datasets peft accelerate bitsandbytes trl

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

In [ ]:
from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Tuple

import torch
from datasets import load_from_disk
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl.trainer import SFTConfig, SFTTrainer


MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Qwen chat tokens (match training/inference format)
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"


def format_example(example: dict) -> dict:
    instruction = (example.get("query") or "").strip()
    instruction = instruction.replace("(use the provided format with backticks)", "")
    instruction = instruction.replace("and enclose your code within delimiters.", "")
    instruction = instruction.rstrip()
    instruction += "\n\nRespond with only the Python code. No explanations, no markdown."

    # Split them up instead of concatenating them
    prompt = f"{CHAT_USER}{instruction}{CHAT_END}\n{CHAT_ASSISTANT}"
    completion = f"{example['completion']}{CHAT_END}"

    return {"prompt": prompt, "completion": completion}



def load_model_and_tokenizer(
    model_id: str,
    use_4bit: bool = True,
    use_flash_attention: bool = False,
):
    """Load Qwen2.5-Coder with optional 4-bit quantization."""

    # Quantization config for memory efficiency
    bnb_config = None
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        padding_side="left",
    )



    # Set pad token if not present
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id



    # Model loading kwargs (low_cpu_mem_usage reduces RAM spikes during load)
    model_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float16,
        "device_map": {"": 0},                 # force everything to GPU 0
        "low_cpu_mem_usage": True,
        "max_memory": {0: "9.5GiB", "cpu": "40GiB"},  # adjust GPU GiB to your 5070 VRAM
    }

    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config

    if use_flash_attention:
        model_kwargs["attn_implementation"] = "flash_attention_2"

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.config.pad_token_id = tokenizer.eos_token_id
    model.config.use_cache = False  # important w/ grad checkpointing

    # Prepare model for k-bit training if using quantization
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
        model.enable_input_require_grads()


    return model, tokenizer


def check_bf16_support() -> Tuple[bool, str]:
    """
    Check if the current setup supports bfloat16.

    Returns:
        Tuple of (supports_bf16: bool, info_message: str)
    """
    info_lines = []

    # Check CUDA availability
    if not torch.cuda.is_available():
        return False, "CUDA is not available"

    info_lines.append(f"CUDA available: {torch.cuda.is_available()}")
    info_lines.append(f"CUDA version: {torch.version.cuda}")
    info_lines.append(f"PyTorch version: {torch.__version__}")

    # Check GPU info
    try:
        device_count = torch.cuda.device_count()
        info_lines.append(f"GPU count: {device_count}")

        for i in range(device_count):
            gpu_name = torch.cuda.get_device_name(i)
            device_capability = torch.cuda.get_device_capability(i)
            info_lines.append(f"GPU {i}: {gpu_name} (Compute Capability: {device_capability[0]}.{device_capability[1]})")

            # Check if compute capability supports bf16 (Ampere 8.0+ or Ada Lovelace 8.9+)
            if device_capability[0] >= 8:
                info_lines.append(f"  -> GPU {i} architecture supports bf16 (Ampere/Ada Lovelace)")
            else:
                info_lines.append(f"  -> GPU {i} architecture may not support bf16 (pre-Ampere)")

        # Try to create a bf16 tensor to test actual support
        try:
            test_tensor = torch.tensor([1.0], dtype=torch.bfloat16, device="cuda:0")
            info_lines.append("bf16 tensor creation test: SUCCESS")
            supports_bf16 = True
        except Exception as e:
            info_lines.append(f"bf16 tensor creation test: FAILED - {e}")
            supports_bf16 = False

    except Exception as e:
        info_lines.append(f"Error checking GPU: {e}")
        supports_bf16 = False

    info_message = "\n".join(info_lines)
    return supports_bf16, info_message


def create_lora_config() -> LoraConfig:
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=64,                          # Increased rank for coding tasks
        lora_alpha=64,                 # Scaling factor of 2.0 (32 / 16)
        lora_dropout=0.05,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        bias="none",
    )

def main(args):
    import gc
    print(f"Loading dataset from {args.data_dir}...")
    dataset = load_from_disk(args.data_dir)
    gc.collect()

    # 1. Format dataset into "prompt" and "completion" columns (num_proc=1 to limit memory)
    print("Formatting dataset...")
    train_dataset = dataset["train"].map(
        format_example,
        remove_columns=dataset["train"].column_names,
        desc="Formatting train",
        num_proc=1,
    )
    eval_dataset = dataset["test"].map(
        format_example,
        remove_columns=dataset["test"].column_names,
        desc="Formatting eval",
        num_proc=1,
    )

    if args.max_train_samples:
        train_dataset = train_dataset.select(range(min(args.max_train_samples, len(train_dataset))))
    if args.max_eval_samples:
        eval_dataset = eval_dataset.select(range(min(args.max_eval_samples, len(eval_dataset))))

    # 2. Load Model & Tokenizer
    print(f"Loading model: {MODEL_ID}...")
    model, tokenizer = load_model_and_tokenizer(
        MODEL_ID,
        use_4bit=args.use_4bit,
        use_flash_attention=args.use_flash_attention,
    )

    # 3. Apply LoRA
    print("Applying LoRA adapters...")
    lora_config = create_lora_config()
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Determine mixed precision settings
    supports_bf16, _ = check_bf16_support()
    use_bf16 = supports_bf16
    use_fp16 = not supports_bf16 and torch.cuda.is_available()

    model.config.use_cache = False  # REQUIRED when using gradient checkpointing
    model.gradient_checkpointing_enable()

    # 4. Configure Trainer natively
    output_dir = Path(args.output_dir)
    config_kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": args.epochs,
        "per_device_train_batch_size": args.batch_size,
        "per_device_eval_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.gradient_accumulation_steps,
        "learning_rate": args.learning_rate, # Should be 2e-4
        "weight_decay": 0.01,
        "warmup_ratio": 0.03,
        "lr_scheduler_type": "cosine",
        "logging_steps": 10,
        "eval_strategy": "steps",
        "eval_steps": args.eval_steps,
        "save_strategy": "steps",
        "save_steps": args.save_steps,
        "save_total_limit": 3,
        "load_best_model_at_end": False,
        "metric_for_best_model": "eval_loss",
        "gradient_checkpointing": args.gradient_checkpointing,
        "optim": "paged_adamw_8bit" if args.use_4bit else "adamw_torch",
        "report_to": "none",
        "push_to_hub": False,
        "max_grad_norm": 1.0,
        "dataloader_num_workers": 0,
        "save_safetensors": True,
        # MODERN TRL CONFIGURATION:
        "completion_only_loss": True,         # Automatically handles -100 masking
        "max_length": args.max_seq_length # Automatically handles truncation
    }

    if use_bf16:
        config_kwargs["bf16"] = True
    elif use_fp16:
        config_kwargs["fp16"] = True

    if args.gradient_checkpointing:
        config_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    sft_config = SFTConfig(**config_kwargs)

    # 5. Initialize Trainer
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    # ==========================================
    # DEBUG: VERIFY COMPLETION-ONLY MASKING
    # ==========================================
    # print("\n--- Running Label Masking Sanity Check ---")
    # train_dataloader = trainer.get_train_dataloader()
    # first_batch = next(iter(train_dataloader))

    # input_ids = first_batch["input_ids"][0]
    # labels = first_batch["labels"][0]

    # total_tokens = len(labels)
    # masked_tokens = (labels == -100).sum().item()
    # trained_tokens = total_tokens - masked_tokens

    # print(f"Total tokens in sequence: {total_tokens}")
    # print(f"Masked prompt tokens (-100): {masked_tokens} ({masked_tokens/total_tokens:.1%})")
    # print(f"Completion tokens to train on: {trained_tokens} ({trained_tokens/total_tokens:.1%})\n")

    # if masked_tokens > 0:
    #     valid_input_ids = input_ids[labels != -100]
    #     decoded_training_target = tokenizer.decode(valid_input_ids[:50])
    #     print(f"First 50 trained tokens: [{decoded_training_target}...]")
    # else:
    #      print("⚠️ WARNING: 0 tokens are masked!")
    # print("------------------------------------------\n")

    # # 6. Train
    print("Starting training...")
    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    # 7. Save
    final_model_path = output_dir / "final"
    print(f"Saving final model to {final_model_path}...")
    trainer.save_model(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))

    print("Training complete!")

# Replace argparse with direct assignments for Colab (tuned for low memory)
class Args:
    def __init__(self):
        self.data_dir = "data/leetcode"
        self.output_dir = "outputs/qwen-lora"
        self.max_train_samples = None   # limit to reduce RAM; set None for full
        self.max_eval_samples = None
        self.use_4bit = True
        self.use_flash_attention = False
        self.epochs = 3
        self.batch_size = 1            # minimal per-step memory
        self.gradient_accumulation_steps = 16   # effective batch = 16
        self.learning_rate = 2e-4      # INCREASED for LoRA
        self.max_seq_length = 3072      # lower = less memory (was 3072)
        self.gradient_checkpointing = True
        self.eval_steps = 50
        self.save_steps = 50
        self.resume_from_checkpoint = False

args = Args()
main(args)

Loading dataset from data/leetcode...
Formatting dataset...
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct...
Applying LoRA adapters...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660
Starting training...


RuntimeError: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: copying a param with shape torch.Size([32, 1536]) from checkpoint, the shape in current model is torch.Size([64, 1536]).
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: copying a param with shape torch.Size([1536, 32]) from checkpoint, the shape in current model is torch.Size([1536, 64]).
	size mismatch for base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: copying a param with shape torch.Size([32, 1536]) from checkpoint, the shape in current model is torch.Size([64, 1536]).
	size mismatch for base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: copying a param with shape torch.Size([256, 32]) from checkpoint, the shape in current model is torch.Size([256, 64]).
	size mismatch for base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.

In [8]:
import ast
import subprocess
import sys
import tempfile
import os
from pathlib import Path
from typing import Tuple, Optional

def check_compilation(code: str) -> Tuple[bool, Optional[str]]:
    """
    Check if Python code compiles without syntax errors.

    Args:
        code: Python code string to check

    Returns:
        Tuple of (compiles: bool, error_message: Optional[str])
    """
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"SyntaxError: {e.msg} at line {e.lineno}"
    except Exception as e:
        return False, f"ParseError: {str(e)}"


def extract_code_from_completion(completion: str, starter_code: str = "") -> str:
    """
    Extract Python code from model completion.

    This function attempts to extract the actual code from the model's output,
    which might include markdown code blocks, explanations, etc.

    Args:
        completion: Model's raw completion text
        starter_code: Optional starter code to prepend

    Returns:
        Extracted Python code string
    """
    # Strip Qwen-style chat markers and special tokens if they leak into the output
    for marker in (
        "<|im_start|>system",
        "<|im_start|>user",
        "<|im_start|>assistant",
        "<|im_start|>",
        "<|im_end|>",
        "<|endoftext|>",
    ):
        completion = completion.replace(marker, "")

    # Truncate at Qwen FIM/file tokens and common failure modes
    # "FRINGEMENT" and "{lng" were observed in diagnostic output
    stop_tokens = [
        "<|file_sep|>",
        "<|fim_prefix|>",
        "<|fim_suffix|>",
        "<|fim_middle|>",
        "<|repo_name|>",
        "FRINGEMENT",
        "\nuser\n", # often hallucinated start of next turn
        "\n{lng",
    ]

    for stop_token in stop_tokens:
        if stop_token in completion:
            completion = completion[:completion.find(stop_token)]

    # Remove markdown code blocks if present
    if "```python" in completion:
        # Extract code between ```python and ```
        start_idx = completion.find("```python") + len("```python")
        end_idx = completion.find("```", start_idx)
        if end_idx != -1:
            completion = completion[start_idx:end_idx].strip()
        else:
             # If no closing tick, take everything after start
            completion = completion[start_idx:].strip()
    elif "```" in completion:
        # Generic code block
        start_idx = completion.find("```") + 3
        end_idx = completion.find("```", start_idx)
        if end_idx != -1:
            completion = completion[start_idx:end_idx].strip()
        else:
            completion = completion[start_idx:].strip()

    # Post-process: Add typing imports if type annotations are used
    extracted_code = completion.strip()

    # Check if code uses type annotations that need imports
    needs_typing = any(annotation in extracted_code for annotation in
                       ['List[', 'Dict[', 'Tuple[', 'Optional[', 'Any', 'Union[', 'Set['])

    if needs_typing and 'from typing import' not in extracted_code:
        # Determine which typing imports are needed
        needed_imports = []
        if 'List[' in extracted_code:
            needed_imports.append('List')
        if 'Dict[' in extracted_code:
            needed_imports.append('Dict')
        if 'Tuple[' in extracted_code:
            needed_imports.append('Tuple')
        if 'Optional[' in extracted_code:
            needed_imports.append('Optional')
        if 'Union[' in extracted_code:
            needed_imports.append('Union')
        if 'Set[' in extracted_code:
            needed_imports.append('Set')
        if 'Any' in extracted_code and 'Any' not in needed_imports:
            needed_imports.append('Any')

        if needed_imports:
            typing_import = f"from typing import {', '.join(needed_imports)}\n"
            extracted_code = typing_import + extracted_code

    # Add common stdlib imports if used but not imported
    stdlib_imports = []
    if 'defaultdict' in extracted_code and 'from collections' not in extracted_code:
        stdlib_imports.append('from collections import defaultdict, deque, Counter')
    elif 'deque' in extracted_code and 'from collections' not in extracted_code:
        stdlib_imports.append('from collections import deque')
    elif 'Counter' in extracted_code and 'from collections' not in extracted_code:
        stdlib_imports.append('from collections import Counter')

    if 'heappush' in extracted_code or 'heappop' in extracted_code:
        if 'from heapq' not in extracted_code and 'import heapq' not in extracted_code:
            stdlib_imports.append('from heapq import heappush, heappop, heapify')

    if ' inf' in extracted_code or '(inf' in extracted_code or '[inf' in extracted_code:
        if 'inf = ' not in extracted_code and 'from math import inf' not in extracted_code:
            stdlib_imports.append('from math import inf')

    if stdlib_imports:
        extracted_code = '\n'.join(stdlib_imports) + '\n' + extracted_code

    # Combine with starter code if provided
    if starter_code:
        # Extract imports from starter code that might be needed
        starter_lines = starter_code.split('\n')
        starter_imports = [line for line in starter_lines
                          if line.strip().startswith(('import ', 'from '))]

        # Check if extracted code already has a class or function definition
        # If so, don't prepend starter code (model provided complete solution)
        has_class_def = 'class ' in extracted_code
        has_func_def = 'def ' in extracted_code

        # Only prepend starter code if the model output doesn't have its own structure
        if not has_class_def and not has_func_def:
            # Check if we need to add starter imports
            if starter_imports:
                extracted_lines = extracted_code.split('\n')
                existing_imports = [line for line in extracted_lines
                                  if line.strip().startswith(('import ', 'from '))]
                for imp in starter_imports:
                    if imp not in existing_imports:
                        extracted_code = imp + '\n' + extracted_code

            # Prepend starter code since model only output function body
            if starter_code.strip() not in extracted_code:
                combined = starter_code + "\n" + extracted_code
                return combined

    return extracted_code


def run_tests(code: str, test_code: str, entry_point: str = "candidate", timeout: int = 10) -> Tuple[bool, Optional[str]]:
    """
    Execute code and run tests in a subprocess with timeout.

    The test_code should contain a check(candidate) function that tests the solution.
    We'll call check(entry_point) where entry_point is the function name from the dataset.

    Args:
        code: The solution code to test
        test_code: The test code containing check(candidate) function
        entry_point: The name of the function to test (default: "candidate")
        timeout: Maximum execution time in seconds

    Returns:
        Tuple of (tests_passed: bool, error_message: Optional[str])
    """
    # Combine code and tests
    # The test_code contains check(candidate), so we need to call it with the entry_point function
    full_code = code + "\n\n" + test_code + f"\n\n# Run the tests\ncheck({entry_point})"
     # ADD THIS - print first failing case
    print("ENTRY POINT:", entry_point)
    print("TOP-LEVEL NAMES DEFINED:")
    import ast
    tree = ast.parse(code)
    top_level = [n.name for n in ast.walk(tree) 
                 if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    print(top_level)  # e.g. ['Solution'] — entry_point won't be in here
    # Create a temporary file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(full_code)
        temp_file = f.name

    try:
        # Run the code in a subprocess with timeout
        result = subprocess.run(
            [sys.executable, temp_file],
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=os.path.dirname(temp_file)
        )

        # Check if tests passed (exit code 0 typically means success)
        if result.returncode == 0:
            return True, None
        else:
            error_msg = result.stderr or result.stdout
            return False, error_msg[:500]  # Limit error message length

    except subprocess.TimeoutExpired:
        return False, f"Timeout after {timeout} seconds"
    except Exception as e:
        return False, f"Execution error: {str(e)}"
    finally:
        # Clean up temp file
        try:
            os.unlink(temp_file)
        except:
            pass

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_from_disk
from tqdm import tqdm
import logging
from pathlib import Path
from typing import Dict

# The functions check_compilation, extract_code_from_completion, and run_tests
# are assumed to be defined in the previous cell or imported.


# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Qwen chat tokens (match training format)
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"


def build_model_prompt(row: dict) -> str:
    # Match training format_example: query only, same replacements and suffix
    user_content = (row.get("query") or "").strip()
    user_content = user_content.replace("(use the provided format with backticks)", "")
    user_content = user_content.replace("and enclose your code within delimiters.", "")
    user_content = user_content.rstrip()
    user_content += "\n\nRespond with only the Python code. No explanations, no markdown."
    return f"{CHAT_USER}{user_content}{CHAT_END}\n{CHAT_ASSISTANT}"


def load_lora_model(base_model_id: str, lora_model_path: str, use_4bit: bool = True):
    """
    Load the base model and apply LoRA adapters.

    Args:
        base_model_id: HuggingFace model identifier for the base model
        lora_model_path: Path to the finetuned LoRA adapters
        use_4bit: Whether to use 4-bit quantization

    Returns:
        Tuple of (model, tokenizer)
    """
    from transformers import BitsAndBytesConfig

    logger.info(f"Loading base model: {base_model_id}")

    # Quantization config for memory efficiency (same as training)
    bnb_config = None
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_id,
        trust_remote_code=True,
        padding_side="right",
    )

    # Set pad token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id


    # Load base model
    model_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.float16,
        "device_map": "auto",
    }

    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config

    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **model_kwargs)

    # Load LoRA adapters
    logger.info(f"Loading LoRA adapters from: {lora_model_path}")
    model = PeftModel.from_pretrained(base_model, lora_model_path)
    model.config.pad_token_id = tokenizer.eos_token_id
    print(type(model))
    print(model.peft_config)

    # Merge adapters for faster inference (optional - comment out if you want to keep them separate)
    # logger.info("Merging LoRA adapters for faster inference...")
    # model = model.merge_and_unload()

    logger.info("Model loaded successfully")
    return model, tokenizer

def clean_code(text: str) -> str:
    # Keep only first class Solution block
    start = text.find("class Solution:")
    if start == -1:
        return text.strip()

    text = text[start:]

    # If another class starts, cut it
    second = text.find("\nclass Solution:", 10)
    if second != -1:
        text = text[:second]

    # Remove trailing partial word after return
    lines = text.splitlines()
    if lines:
        last = lines[-1]
        # If last line has no indentation but contains no valid Python structure
        if (
            not last.startswith(" ")
            and not last.startswith("\t")
            and "class" not in last
            and "def" not in last
            and "=" not in last
        ):
            lines = lines[:-1]

    return "\n".join(lines).strip()


def generate_code(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 256,   # lower is safer
    do_sample: bool = False,
    temperature: float = 0.2,
) -> str:

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # === STOP TOKENS ===
    eos_ids = []

    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            eos_ids.append(tid)

    # === BAN FIM / REPO TOKENS ===
    banned = [
        "<|fim_prefix|>",
        "<|fim_middle|>",
        "<|fim_suffix|>",
        "<|fim_pad|>",
        "<|repo_name|>",
    ]

    bad_words_ids = []
    for tok in banned:
        ids = tokenizer.encode(tok, add_special_tokens=False)
        if ids:
            bad_words_ids.append(ids)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=eos_ids,
        bad_words_ids=bad_words_ids,
    )

    # Remove None entries
    gen_kwargs = {k: v for k, v in gen_kwargs.items() if v is not None}

    model.eval()
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)

    # === HARD CUT ON STOP MARKERS ===
    for marker in [
        "<|im_end|>",
        "<|endoftext|>",
        "<|repo_name|>",
        "<|fim_prefix|>",
        "<|fim_middle|>",
        "<|fim_suffix|>",
        "<|fim_pad|>",
    ]:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]

    # === KEEP ONLY FIRST class Solution BLOCK ===
    first = text.find("class Solution:")
    if first != -1:
        text = text[first:]
        second = text.find("\nclass Solution:", 10)
        if second != -1:
            text = text[:second]

    return text.strip()

def evaluate_on_dataset(
    model,
    tokenizer,
    dataset,
    max_samples: int = None,
    max_new_tokens: int = 512,
    temperature: float = 0.0,
) -> Dict[str, float]:
    """
    Evaluate model on test dataset.

    Args:
        model: The language model
        tokenizer: The tokenizer
        dataset: Test dataset
        max_samples: Maximum number of samples to evaluate (None for all)
        max_new_tokens: Maximum number of tokens to generate
        temperature: Sampling temperature for generation

    Returns:
        Dictionary with metrics: compile_rate, test_pass_rate, total_samples
    """
    test_split = dataset["test"]
    total_samples = len(test_split) if max_samples is None else min(max_samples, len(test_split))

    logger.info(f"Evaluating on {total_samples} samples from test set")

    compile_count = 0
    test_pass_count = 0
    total_evaluated = 0

    for idx in tqdm(range(total_samples), desc="Evaluating"):

        row = test_split[idx]

        # Build prompt (matches training format_example)
        prompt = build_model_prompt(row)
        starter_code = row.get("starter_code", "")
        test_code = row.get("test", "")
        entry_point = row.get("entry_point", "candidate")
        task_id = row.get("task_id", f"task_{idx}")

        if idx < 5:
          print(f"Prompt: {prompt}")
          print(f"Starter Code: {starter_code}")

        if not prompt.strip() or not test_code:
            logger.warning(f"Skipping sample {idx}: missing query/prompt or test code")
            continue

        # Generate code
        try:
            generated_completion = clean_code(generate_code(
                model,
                tokenizer,
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=temperature,
            ))
            if idx < 5:
                print(f"\nGenerated Completion: {generated_completion}\n")
        except Exception as e:
            logger.error(f"Error generating code for {task_id}: {e}")
            continue

        # Extract code from completion
        extracted_code = extract_code_from_completion(generated_completion, starter_code)

        # Full runnable code: dataset "prompt" has required imports/context (see testing_script.py).
        # Without it, tests fail with NameError (List, Dict, etc.) or missing context.
        prelude = (row.get("prompt") or "").strip()
        full_solution = (prelude + "\n\n" + extracted_code).strip() if prelude else extracted_code

        compiles, compile_error = check_compilation(full_solution)

        if compiles:
            compile_count += 1
        else:
            logger.debug(f"Compilation failed for {task_id}: {compile_error}")

        # Run tests (only if code compiles)
        tests_passed = False
        if compiles:
            try:
                tests_passed, test_error = run_tests(full_solution, test_code, entry_point=entry_point, timeout=10)
                if tests_passed:
                    test_pass_count += 1
                else:
                    logger.debug(f"Tests failed for {task_id}: {test_error}")
            except Exception as e:
                logger.debug(f"Error running tests for {task_id}: {e}")

        total_evaluated += 1

        # Log progress periodically
        if (idx + 1) % 10 == 0:
            current_compile_rate = (compile_count / total_evaluated) * 100
            current_test_rate = (test_pass_count / total_evaluated) * 100
            logger.info(
                f"Progress: {idx + 1}/{total_samples} | "
                f"Compile rate: {current_compile_rate:.2f}% | "
                f"Test pass rate: {current_test_rate:.2f}%"
            )

    # Calculate final metrics
    compile_rate = (compile_count / total_evaluated) * 100 if total_evaluated > 0 else 0.0
    test_pass_rate = (test_pass_count / total_evaluated) * 100 if total_evaluated > 0 else 0.0

    return {
        "compile_rate": compile_rate,
        "test_pass_rate": test_pass_rate,
        "compile_count": compile_count,
        "test_pass_count": test_pass_count,
        "total_evaluated": total_evaluated
    }


def main():
    """Main evaluation function."""
    # Replace argparse with direct assignments for Colab
    model_path = "outputs/qwen-lora/final"
    base_model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    data_dir = "data/leetcode"
    max_samples = None
    use_4bit = True
    max_new_tokens = 512
    temperature = 0.2

    # Load model
    model, tokenizer = load_lora_model(base_model_id, model_path, use_4bit=use_4bit)

    # Load dataset
    logger.info(f"Loading dataset from {data_dir}...")
    dataset = load_from_disk(data_dir)
    logger.info(f"Dataset loaded. Test set size: {len(dataset['test'])}")

    # Evaluate
    metrics = evaluate_on_dataset(
        model,
        tokenizer,
        dataset,
        max_samples=max_samples,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )

    # Log final results
    logger.info("=" * 60)
    logger.info("FINAL EVALUATION RESULTS")
    logger.info("=" * 60)
    logger.info(f"Model path: {model_path}")
    logger.info(f"Base model: {base_model_id}")
    logger.info(f"Total samples evaluated: {metrics['total_evaluated']}")
    logger.info(f"Compilation rate: {metrics['compile_rate']:.2f}% ({metrics['compile_count']}/{metrics['total_evaluated']})")
    logger.info(f"Test pass rate: {metrics['test_pass_rate']:.2f}% ({metrics['test_pass_count']}/{metrics['total_evaluated']})")
    logger.info("=" * 60)

    print("=" * 60)
    print("FINAL EVALUATION RESULTS")
    print("=" * 60)
    print(f"Model path: {model_path}")
    print(f"Base model: {base_model_id}")
    print(f"Total samples evaluated: {metrics['total_evaluated']}")
    print(f"Compilation rate: {metrics['compile_rate']:.2f}% ({metrics['compile_count']}/{metrics['total_evaluated']})")
    print(f"Test pass rate: {metrics['test_pass_rate']:.2f}% ({metrics['test_pass_count']}/{metrics['total_evaluated']})")
    print("=" * 60)


main()

2026-03-06 18:27:34,568 - INFO - Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct


2026-03-06 18:27:35,136 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2026-03-06 18:27:36,823 - INFO - Loading LoRA adapters from: outputs/qwen-lora/final
2026-03-06 18:27:37,468 - INFO - Model loaded successfully
2026-03-06 18:27:37,469 - INFO - Loading dataset from data/leetcode...
2026-03-06 18:27:37,483 - INFO - Dataset loaded. Test set size: 228
2026-03-06 18:27:37,483 - INFO - Evaluating on 228 samples from test set


<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path='Qwen/Qwen2.5-Coder-1.5B-Instruct', revision=None, inference_mode=True, r=32, target_modules={'o_proj', 'k_proj', 'q_proj', 'up_proj', 'v_proj', 'gate_proj', 'down_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)}


Evaluating:   0%|          | 0/228 [00:00<?, ?it/s]

Prompt: <|im_start|>user
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
You are given an integer n and a 2D integer array queries.
There are n cities numbered from 0 to n - 1. Initially, there is a unidirectional road from city i to city i + 1 for all 0 <= i < n - 1.
queries[i] = [ui, vi] represents the addition of a new unidirectional road from city ui to city vi. After each query, you need to find the length of the shortest path from city 0 to city n - 1.
Return an array answer where for each i in the range [0, queries.length - 1], answer[i] is the length of the shortest path from city 0 to city n - 1 after processing the first i + 1 queries.
 
Example 1:

Input: n = 5, queries = [[2,4],[0,2],[0,4]]
Output: [3,2,1]
Explanation: 

After the addition of the road from 2 to 4, the length of the shortest path from 0 to 4 is 3.

After the

...


Generated Completion: class Solution:
    def shortestDistanceAfterQueries(
        self, n: int, queries: List[List[int]]
    ) -> List[int]:
        g = [[] for _ in range(n)]
        for u, v in pairwise(range(n - 1)):
            g[u].append(v)
            g[v].append(u)
        dist = [inf] * n
        dist[0] = 0
        ans = []
        for u, v in queries:
            q = deque([u])
            vis = {u}
            while q:
                i = q.popleft()
                for j in g[i]:
                    if j not in vis:
                        vis.add(j)
                        q.append(j)
                        dist[j] = min(dist[j], dist[i] + 1)
            ans.append(dist[n - 1])
        return ans

ENTRY POINT: Solution().shortestDistanceAfterQueries
TOP-LEVEL NAMES DEFINED:
['ListNode', 'list_node', 'is_same_list', 'TreeNode', 'tree_node', 'is_same_tree', 'Solution', '__init__', '__init__', 'shortestDistanceAfterQueries']
Prompt: <|im_start|>user
You are an expert Pyt

Evaluating:   1%|▏         | 3/228 [00:26<29:27,  7.86s/it]


Generated Completion: class Solution:
    def subsequenceCount(self, nums: List[int]) -> int:
        mod = 10**9 + 7
        cnt = Counter(x % 2 for x in nums)
        ans = pow(2, cnt[1], mod)
        ans *= cnt[0]
        ans %= mod
        return ans

ENTRY POINT: Solution().subsequenceCount
TOP-LEVEL NAMES DEFINED:
['ListNode', 'list_node', 'is_same_list', 'TreeNode', 'tree_node', 'is_same_tree', 'Solution', '__init__', '__init__', 'subsequenceCount']
Prompt: <|im_start|>user
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

### Question:
There is a snake in an n x n matrix grid and can move in four possible directions. Each cell in the grid is identified by the position: grid[i][j] = (i * n) + j.
The snake starts at cell 0 and follows a sequence of commands.
You are given an integer n representing the size of the grid and an array of strings co

Evaluating:   1%|▏         | 3/228 [00:28<35:22,  9.43s/it]


KeyboardInterrupt: 

In [ ]:
import numpy as np
from tqdm import tqdm

def analyze_dataset_lengths(dataset, tokenizer, max_seq_length=1024, min_supervised_tokens=64):
    """
    Analyze how many examples will be truncated or poorly supervised.
    """

    prompt_lengths = []
    response_lengths = []
    total_lengths = []
    supervised_after_trunc = []

    for example in tqdm(dataset):
        prompt_ids = tokenizer(
            example["prompt"],
            add_special_tokens=False
        )["input_ids"]

        # Training uses "completion"; fallback to "response" for older datasets
        response_text = example.get("completion", example.get("response", ""))
        response_ids = tokenizer(
            response_text,
            add_special_tokens=False
        )["input_ids"]

        prompt_len = len(prompt_ids)
        response_len = len(response_ids)
        total_len = prompt_len + response_len

        # simulate truncation
        truncated_total = min(total_len, max_seq_length)

        # supervised tokens are response tokens that survive truncation
        supervised_tokens = max(
            0,
            truncated_total - prompt_len
        )

        prompt_lengths.append(prompt_len)
        response_lengths.append(response_len)
        total_lengths.append(total_len)
        supervised_after_trunc.append(supervised_tokens)

    prompt_lengths = np.array(prompt_lengths)
    response_lengths = np.array(response_lengths)
    total_lengths = np.array(total_lengths)
    supervised_after_trunc = np.array(supervised_after_trunc)

    print("\n========== DATASET LENGTH ANALYSIS ==========\n")

    print(f"Total examples: {len(dataset)}")
    print(f"Max sequence length: {max_seq_length}\n")

    print("---- Prompt Length ----")
    print(f"Mean: {prompt_lengths.mean():.1f}")
    print(f"95th percentile: {np.percentile(prompt_lengths, 95):.1f}")
    print(f"Max: {prompt_lengths.max()}\n")

    print("---- Response Length ----")
    print(f"Mean: {response_lengths.mean():.1f}")
    print(f"95th percentile: {np.percentile(response_lengths, 95):.1f}")
    print(f"Max: {response_lengths.max()}\n")

    print("---- Total Length ----")
    print(f"Mean: {total_lengths.mean():.1f}")
    print(f"95th percentile: {np.percentile(total_lengths, 95):.1f}")
    print(f"Max: {total_lengths.max()}\n")

    too_long = (total_lengths > max_seq_length).sum()
    print(f"Examples exceeding max_seq_length: {too_long} "
          f"({100*too_long/len(dataset):.2f}%)")

    no_supervision = (supervised_after_trunc == 0).sum()
    print(f"Examples with ZERO supervised tokens after truncation: "
          f"{no_supervision} ({100*no_supervision/len(dataset):.2f}%)")

    low_supervision = (supervised_after_trunc < min_supervised_tokens).sum()
    print(f"Examples with <{min_supervised_tokens} supervised tokens: "
          f"{low_supervision} ({100*low_supervision/len(dataset):.2f}%)")

    print("\n=============================================\n")

    return {
        "prompt_lengths": prompt_lengths,
        "response_lengths": response_lengths,
        "total_lengths": total_lengths,
        "supervised_after_trunc": supervised_after_trunc,
    }

data_dir = "/content/drive/MyDrive/PyPilot/data/leetcode"
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    padding_side="left",
)

# Load dataset
logger.info(f"Loading dataset from {data_dir}...")
dataset = load_from_disk(data_dir)
analysis = analyze_dataset_lengths(
    dataset["train"],
    tokenizer,
    max_seq_length=3072,
    min_supervised_tokens=64,
)



  0%|          | 0/2641 [00:00<?, ?it/s]

  1%|▏         | 37/2641 [00:00<00:07, 369.61it/s]

  3%|▎         | 76/2641 [00:00<00:06, 376.05it/s]

  4%|▍         | 114/2641 [00:00<00:06, 374.57it/s]

  6%|▌         | 153/2641 [00:00<00:06, 377.52it/s]

  7%|▋         | 191/2641 [00:00<00:06, 376.52it/s]

  9%|▊         | 229/2641 [00:00<00:06, 375.40it/s]

 10%|█         | 267/2641 [00:00<00:06, 369.50it/s]

 12%|█▏        | 304/2641 [00:00<00:06, 367.90it/s]

 13%|█▎        | 342/2641 [00:00<00:06, 369.14it/s]

 14%|█▍        | 380/2641 [00:01<00:06, 369.92it/s]

 16%|█▌        | 417/2641 [00:01<00:06, 363.54it/s]

 17%|█▋        | 455/2641 [00:01<00:05, 365.56it/s]

 19%|█▊        | 492/2641 [00:01<00:05, 365.27it/s]

 20%|██        | 530/2641 [00:01<00:05, 367.16it/s]

 21%|██▏       | 567/2641 [00:01<00:05, 365.69it/s]

 23%|██▎       | 604/2641 [00:01<00:05, 361.23it/s]

 24%|██▍       | 641/2641 [00:01<00:05, 359.50it/s]

 26%|██▌       | 677/2641 [00:01<00:05, 355.46it/s]

 27%


========== DATASET LENGTH ANALYSIS ==========

Total examples: 2641
Max sequence length: 3072

---- Prompt Length ----
Mean: 438.5
95th percentile: 440.0
Max: 453

---- Response Length ----
Mean: 337.7
95th percentile: 692.0
Max: 2048

---- Total Length ----
Mean: 776.2
95th percentile: 1132.0
Max: 2488

Examples exceeding max_seq_length: 0 (0.00%)
Examples with ZERO supervised tokens after truncation: 0 (0.00%)
Examples with <64 supervised tokens: 14 (0.53%)




In [15]:
# ============================================================
# Evaluate BASE Qwen2.5-Coder (NO LoRA)
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_from_disk
from tqdm import tqdm
import logging

# -----------------------
# CONFIG
# -----------------------
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_DIR = "data/leetcode"
USE_4BIT = True  # 4-bit to avoid OOM on limited hardware
MAX_NEW_TOKENS = 512   # lower to limit memory during generation
TEMPERATURE = 0.0
MAX_SAMPLES = 350   # Increase (e.g. 350) for full eval; lower prevents crashes

# -----------------------
# Logging
# -----------------------
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -----------------------
# Chat tokens (must match training format)
# -----------------------
CHAT_USER = "<|im_start|>user\n"
CHAT_ASSISTANT = "<|im_start|>assistant\n"
CHAT_END = "<|im_end|>"

# -----------------------
# Prompt builder
# -----------------------
def build_model_prompt(row: dict) -> str:
    # Match training format_example exactly: query + same replacements + rstrip + suffix
    user_content = (row.get("query") or "").strip()
    user_content = user_content.replace("(use the provided format with backticks)", "")
    user_content = user_content.replace("and enclose your code within delimiters.", "")
    user_content = user_content.rstrip()
    user_content += "\n\nRespond with only the Python code. No explanations, no markdown."
    return f"{CHAT_USER}{user_content}{CHAT_END}\n{CHAT_ASSISTANT}"

# -----------------------
# Generation
# -----------------------
def generate_code(model, tokenizer, prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    eos_ids = []
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            eos_ids.append(tid)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_ids
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)

    for marker in ["<|im_end|>", "<|endoftext|>"]:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]

    # Keep only first class Solution block (match LoRA eval pipeline)
    first = text.find("class Solution:")
    if first != -1:
        text = text[first:]
        second = text.find("\nclass Solution:", 10)
        if second != -1:
            text = text[:second]

    return text.strip()

# -----------------------
# Import your harness utilities
# (These must already exist in your notebook)
# -----------------------
# check_compilation
# extract_code_from_completion
# run_tests

# -----------------------
# Evaluation
# -----------------------
def evaluate_base_model():

    logger.info("Loading base model...")

    bnb_config = None
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        padding_side="left",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
        "low_cpu_mem_usage": True,
    }

    if bnb_config:
        model_kwargs["quantization_config"] = bnb_config

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)

    dataset = load_from_disk(DATA_DIR)
    test_split = dataset["test"]

    total_samples = len(test_split) if MAX_SAMPLES is None else min(MAX_SAMPLES, len(test_split))

    compile_count = 0
    test_pass_count = 0

    for idx in tqdm(range(total_samples), desc="Evaluating Base Model"):

        row = test_split[idx]

        prompt = build_model_prompt(row)
        starter_code = row.get("starter_code", "")
        test_code = row.get("test", "")
        entry_point = row.get("entry_point", "candidate")

        if not prompt.strip() or not test_code:
            continue

        try:
            generated = generate_code(model, tokenizer, prompt)
            if idx < 5:
                print(f"\nGenerated Completion: {generated}\n")
        except Exception:
            continue

        extracted = extract_code_from_completion(generated, starter_code)

        # Prepend dataset "prompt" (imports/context) so tests have List, Dict, etc. defined
        prelude = (row.get("prompt") or "").strip()
        full_solution = (prelude + "\n\n" + extracted).strip() if prelude else extracted

        compiles, _ = check_compilation(full_solution)

        if compiles:
            compile_count += 1
            passed, _ = run_tests(full_solution, test_code, entry_point=entry_point, timeout=10)
            if passed:
                test_pass_count += 1

    compile_rate = 100 * compile_count / total_samples
    pass_rate = 100 * test_pass_count / total_samples

    print("=" * 60)
    print("BASE MODEL RESULTS")
    print("=" * 60)
    print(f"Total evaluated: {total_samples}")
    print(f"Compile rate: {compile_rate:.2f}% ({compile_count}/{total_samples})")
    print(f"Test pass rate: {pass_rate:.2f}% ({test_pass_count}/{total_samples})")
    print("=" * 60)


# Run evaluation
evaluate_base_model()

2026-03-05 18:29:43,400 - INFO - Loading base model...
Evaluating Base Model:   0%|          | 1/228 [00:05<21:52,  5.78s/it]


Generated Completion: class Solution:
    def shortestDistanceAfterQueries(self, n: int, queries: List[List[int]]) -> List[int]:
        graph = [[] for _ in range(n)]
        for u, v in queries:
            graph[u].append(v)
            graph[v].append(u)

        dist = [n-1] * n
        dist[0] = 0

        for u in range(n):
            queue = [u]
            visited = set([u])
            while queue:
                node = queue.pop(0)
                for neighbor in graph[node]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)
                        dist[neighbor] = min(dist[neighbor], dist[node] + 1)

        return dist
```



Evaluating Base Model:   1%|          | 2/228 [00:11<21:52,  5.81s/it]


Generated Completion: class Solution:
    def shortestDistanceAfterQueries(self, n: int, queries: List[List[int]]) -> List[int]:
        graph = [[] for _ in range(n)]
        for u, v in queries:
            graph[u].append(v)
            graph[v].append(u)

        dist = [float('inf')] * n
        dist[0] = 0
        heap = [(0, 0)]

        while heap:
            d, u = heapq.heappop(heap)
            if d > dist[u]:
                continue
            for v in graph[u]:
                if dist[v] > d + 1:
                    dist[v] = d + 1
                    heapq.heappush(heap, (d + 1, v))

        return dist
```



Evaluating Base Model:   1%|▏         | 3/228 [00:18<24:04,  6.42s/it]

...


Generated Completion: class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        # Initialize the snake's starting position
        row, col = 0, 0
        
        # Define the direction of the snake
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        current_direction_index = 0
        
        # Create a list to store the positions of the snake
        snake_positions = [(row, col)]
        
        for command in commands:
            if command == "UP":
                current_direction_index = (current_direction_index - 1) % 4
            elif command == "DOWN":
                current_direction_index = (current_direction_index + 1) % 4
            elif command == "LEFT":
                col -= 1
            elif command == "RIGHT":
                col += 1
            
            # Check if the new position is out of bounds
            if col < 0 or col >= n or row < 0 or row >= n:
                continue
            
         

Evaluating Base Model:   2%|▏         | 5/228 [00:34<25:14,  6.79s/it]


Generated Completion: class Solution:
    def countGoodNodes(self, edges: List[List[int]]) -> int:
        graph = defaultdict(list)
        for u, v in edges:
            graph[u].append(v)
            graph[v].append(u)

        def dfs(node, parent):
            if not graph[node]:
                return 1

            count = 1
            for neighbor in graph[node]:
                if neighbor != parent:
                    count += dfs(neighbor, node)
            
            return count
        
        return sum(dfs(i, None) for i in range(len(graph)))
```



Evaluating Base Model: 100%|██████████| 228/228 [22:54<00:00,  6.03s/it]

BASE MODEL RESULTS
Total evaluated: 228
Compile rate: 0.44% (1/228)
Test pass rate: 0.00% (0/228)
